In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
            border-radius: 12px; padding: 40px 40px 30px 40px; margin: 10px 0 30px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <div style="color:#a0aec0; font-size:0.8em; letter-spacing:2px; text-transform:uppercase; margin-bottom:12px;">
    LLM Lab — Episode 01
  </div>
  <h1 style="color:white; font-size:2.2em; margin:0 0 10px 0; font-weight:700; line-height:1.2; border:none;">
    3-Model Arena Comparison<br>
    <span style="color:#63b3ed;">on Your Mac</span>
  </h1>
  <p style="color:#cbd5e0; font-size:1em; margin:16px 0 24px 0; max-width:600px; line-height:1.6;">
    Apple Silicon’s unified memory changes everything. Run 122B, 35B, and 0.8B
    models side-by-side to see the quality/speed tradeoff viscerally.
  </p>
  <div style="display:flex; gap:16px; flex-wrap:wrap;">
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🍎 Apple Silicon</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🧠 MLX</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🔌 OpenAI API</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🏆 3 Models</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">⏱ ~20 min</span>
  </div>
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💻 Step 1: Hardware Check</h2>
</div>

First — what are we working with?

In [ ]:
!python3 setup_check.py

In [ ]:
# Connect to 3 MLX servers and warm up in parallel
from openai import OpenAI
import time
import concurrent.futures
from IPython.display import HTML, display

MODELS = [
    {"label": "122B", "model": "arthurcollet/Qwen3.5-122B-A10B-mlx-nvfp4", "port": 8800, "color": "#2563eb"},
    {"label": "35B", "model": "RepublicOfKorokke/Qwen3.5-35B-A3B-mlx-lm-nvfp4", "port": 8801, "color": "#16a34a"},
    {"label": "2B", "model": "RepublicOfKorokke/Qwen3.5-2B-mlx-lm-nvfp4", "port": 8802, "color": "#f59e0b"},
]

# Create clients
clients = {}
for m in MODELS:
    clients[m["label"]] = OpenAI(base_url=f"http://localhost:{m['port']}/v1", api_key="unused")

# Warm up all 3 in parallel
def warmup(m):
    t0 = time.time()
    try:
        clients[m["label"]].chat.completions.create(
            model=m["model"],
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=1,
        )
        return m["label"], time.time() - t0, True
    except Exception as e:
        return m["label"], time.time() - t0, False

print("Warming up 3 models in parallel...", flush=True)
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
    warmup_results = list(ex.map(warmup, MODELS))

# Display status table
rows = ""
for label, elapsed, ok in warmup_results:
    status = "✓" if ok else "✗"
    color = "#16a34a" if ok else "#dc2626"
    m = next(m for m in MODELS if m["label"] == label)
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{label}</td>"
        f"<td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>{m['port']}</td>"
        f"<td style='padding:6px 12px; color:{color}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{status}</td>"
        f"<td style='padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{elapsed:.1f}s</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Port</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Status</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Warmup Time</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))
print("All models ready!")

In [ ]:
# Helper functions for 3-model comparisons
import re
import time
import html
import concurrent.futures
from IPython.display import HTML, display

def strip_think(text):
    """Remove <think>...</think> blocks from Qwen3.5 reasoning output."""
    cleaned = re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL)
    if '<think>' in cleaned:
        cleaned = cleaned[:cleaned.index('<think>')]
    return cleaned.strip()

def _render_cards(results, models_order):
    """Render 3-column HTML cards from results list."""
    cards = ""
    for r in results:
        safe_text = html.escape(r["text"]) if r["text"] else "(no response)"
        cards += f"""
        <div style="flex:1; min-width:250px; background:#f9fafb; border:1px solid #d1d5db;
                    border-left:4px solid {r['color']}; border-radius:0 8px 8px 0; padding:12px 18px;">
          <div style="color:{r['color']}; font-weight:bold; font-size:0.85em; margin-bottom:6px;">
            {r['label']} · {r['tps']:.1f} tok/s
          </div>
          <div style="color:#1f2937; line-height:1.6; white-space:pre-wrap; word-wrap:break-word; font-size:0.9em;">
            {safe_text}
          </div>
          <div style="color:#9ca3af; font-size:0.75em; margin-top:8px;">
            {r['tokens']} tokens in {r['elapsed']:.1f}s
          </div>
        </div>"""
    return f'<div style="display:flex; gap:16px; flex-wrap:wrap; margin:10px 0;">{cards}</div>'

def compare_models(prompt, system_prompt=None, **kwargs):
    """Fire the same prompt at all 3 models in parallel (no streaming), display side-by-side cards."""
    kwargs.setdefault("max_tokens", 1024)

    # Disable Qwen3.5 thinking mode so tokens go to the actual answer
    kwargs.setdefault("extra_body", {"chat_template_kwargs": {"enable_thinking": False}})

    order = {"2B": 0, "35B": 1, "122B": 2}
    models_order = sorted(MODELS, key=lambda m: order.get(m["label"], 99))

    def query_model(m):
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        t0 = time.time()
        try:
            r = clients[m["label"]].chat.completions.create(
                model=m["model"], messages=messages, **kwargs
            )
            elapsed = time.time() - t0
            raw = r.choices[0].message.content or ""
            text = strip_think(raw)
            tokens = r.usage.completion_tokens
            return {
                "label": m["label"], "text": text, "tokens": tokens,
                "elapsed": elapsed, "tps": tokens / elapsed if elapsed > 0 else 0,
                "color": m["color"]
            }
        except Exception as e:
            elapsed = time.time() - t0
            return {
                "label": m["label"], "text": f"Error: {e}", "tokens": 0,
                "elapsed": elapsed, "tps": 0, "color": m["color"]
            }

    # Run all 3 in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
        futures = {ex.submit(query_model, m): m["label"] for m in MODELS}
        results_map = {}
        for f in concurrent.futures.as_completed(futures):
            r = f.result()
            results_map[r["label"]] = r

    # Order: 2B, 35B, 122B
    results = [results_map[m["label"]] for m in models_order]

    display(HTML(_render_cards(results, models_order)))
    return results


def show_metrics_table(results):
    """Render a comparison metrics table from compare_models results."""
    rows = ""
    for r in results:
        rows += (
            f"<tr>"
            f"<td style='padding:6px 12px; color:{r['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['label']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['tokens']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['elapsed']:.1f}s</td>"
            f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['tps']:.1f}</td>"
            f"</tr>"
        )
    display(HTML(f"""
    <table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
      <thead><tr style="background:#1e3a5f;">
        <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Tokens</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Time</th>
        <th style="padding:8px 12px; color:white; text-align:left;">tok/s</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>
    """))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 Step 2: First Inference — 3-Model Comparison</h2>
</div>

In [ ]:
results = compare_models(
    "Explain what you are in exactly 3 sentences, as if you were describing yourself to an electrical engineer who has never heard of an LLM.",
)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📊 Step 3: Measure Performance</h2>
</div>

In [ ]:
# Measure tok/s across all 3 models with a fun prompt
import mlx.core as mx

results = compare_models(
    "Write a limerick about UART protocol timing violations.",
)

show_metrics_table(results)

gpu_mem = mx.metal.get_active_memory() / 1e9
print(f"\nActive GPU memory: {gpu_mem:.1f} GB")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧩 Step 4: Memory Topology</h2>
</div>

Why can Apple Silicon run all 3 models simultaneously? **Unified memory.** The CPU, GPU, and Neural Engine share the same RAM — no PCIe bottleneck, no separate VRAM pool.

In [ ]:
# Show the memory topology — why Apple Silicon is special for this
import mlx.core as mx
import psutil
from IPython.display import HTML, display

gpu_memory = mx.metal.get_active_memory() / 1e9
ram = psutil.virtual_memory()

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Architecture</th>
    <th style="padding:8px 12px; color:white; text-align:left;">GPU Memory</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Access</th>
  </tr></thead>
  <tbody>
    <tr><td style="padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;">Discrete GPU</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">24 GB VRAM</td>
        <td style="padding:6px 12px; color:#dc2626; font-weight:bold; border-bottom:1px solid #e5e7eb;">Model must fit here</td></tr>
    <tr><td style="padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;">This Mac</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">{ram.total/1e9:.0f} GB Unified</td>
        <td style="padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;">CPU + GPU + Neural Engine share</td></tr>
  </tbody>
</table>

<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Approx Footprint</th>
  </tr></thead>
  <tbody>
    <tr><td style="padding:6px 12px; color:#2563eb; font-weight:bold; border-bottom:1px solid #e5e7eb;">122B</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">~65 GB</td></tr>
    <tr><td style="padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;">35B</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">~20 GB</td></tr>
    <tr><td style="padding:6px 12px; color:#f59e0b; font-weight:bold; border-bottom:1px solid #e5e7eb;">2B</td>
        <td style="padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;">~1.2 GB</td></tr>
    <tr style="background:#f0f4ff;"><td style="padding:6px 12px; color:#1e3a5f; font-weight:bold;">Total</td>
        <td style="padding:6px 12px; color:#1e3a5f; font-weight:bold;">~86.2 GB</td></tr>
  </tbody>
</table>
"""))

print(f"Active GPU memory: {gpu_memory:.1f} GB")
print(f"System RAM used: {ram.percent:.0f}%")
print(f"Available RAM: {ram.available/1e9:.0f} GB")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🌡️ Step 5: Temperature Experiment</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> Temperature controls the randomness of token sampling. At <code>0</code> the model always picks the most likely next token (deterministic). Higher values spread probability across more tokens, making output more creative — or more chaotic.
</div>

In [ ]:
# Temperature = 0.7 across all 3 models — watch the quality spread
results = compare_models(
    "In one sentence, describe how electricity flows through a wire, but make it sound like ancient mythology.",
    temperature=0.7,
)

# Try other temperatures yourself — change the value above and re-run!

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🎭 Step 6: System Prompts</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> A system prompt sets the model’s persona and behavior rules. The weights don’t change — you’re just framing the input differently. Think of it as a costume for the model.
</div>

In [ ]:
# Pirate persona across all 3 models — entertaining quality spread!
results = compare_models(
    "What is a neural network?",
    system_prompt="You are a pirate who somehow understands machine learning. Arr.",
    temperature=0.8,
)

In [ ]:
# Bonus: 3 personas on the 122B model only
import re
import html as html_mod
from IPython.display import HTML, display

personas = [
    ("Grumpy Firmware Engineer", "\U0001f610", "You are a grumpy firmware engineer with 30 years of experience. You think everything modern is bloated and wrong. Be brief and sarcastic.", "#dc2626"),
    ("Excited Intern", "\U0001f929", "You are an extremely enthusiastic first-year CS student who just discovered transformers. Use lots of exclamation points.", "#2563eb"),
    ("ML Pirate", "\U0001f3f4\u200d\u2620\ufe0f", "You are a pirate who somehow understands machine learning. Arr.", "#16a34a"),
]

question = "What is a neural network?"
model_122b = next(m for m in MODELS if m["label"] == "122B")

cards = ""
for name, icon, system, color in personas:
    r = clients["122B"].chat.completions.create(
        model=model_122b["model"],
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": question},
        ],
        max_tokens=512, temperature=0.8,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    raw = r.choices[0].message.content.strip()
    cleaned = strip_think(raw)
    text = html_mod.escape(cleaned)
    cards += f"""
    <div style="flex:1; min-width:250px; background:#f9fafb; border:1px solid #d1d5db; border-radius:8px;
                padding:16px 20px; font-family:'Inter',sans-serif; word-wrap:break-word; overflow-wrap:break-word;">
      <div style="color:{color}; font-weight:bold; font-size:0.9em; margin-bottom:8px;">
        {icon} {name}
      </div>
      <div style="color:#1f2937; line-height:1.6; white-space:pre-wrap;">{text}</div>
    </div>"""

display(HTML(f"""
<div style="color:#6b7280; font-size:0.85em; margin-bottom:8px;">122B model — same weights, different personas:</div>
<div style="display:flex; gap:16px; flex-wrap:wrap; margin:10px 0;">{cards}</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧮 Step 7: Quantization</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> Quantization reduces the precision of model weights (e.g., from 32-bit floats to 4-bit integers). This shrinks the model ~8× with only a small quality loss, making it possible to fit large models in memory.
</div>

In [ ]:
# What quantization actually does to numbers
import mlx.core as mx
from IPython.display import HTML, display

# Real weights are fp32 — 4 bytes each
weights_fp32 = mx.array([0.1234, -0.5678, 0.9012, -0.3456, 0.7890])

# 4-bit quantization: ~16 possible values per weight, loses precision
# Simulate by rounding to nearest 1/8
weights_4bit_approx = mx.round(weights_fp32 * 8) / 8

print("What quantization does to weights:")
print(f"  fp32:   {weights_fp32.tolist()}")
print(f"  ~4-bit: {weights_4bit_approx.tolist()}")
print()

# Size math — this is why quantization matters
params = 32_000_000_000  # 32B parameters

rows = ""
for label, mult, note in [
    ("fp32 (4 bytes/param)", 4, "doesn't fit on most machines"),
    ("bf16 (2 bytes/param)", 2, "still large"),
    ("4-bit (0.5 bytes/param)", 0.5, "fits in 64GB+ Mac"),
]:
    size = params * mult / 1e9
    rows += (
        f"<tr><td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>{label}</td>"
        f"<td style='padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{size:.0f} GB \u2014 {note}</td></tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Precision</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Size (32B params)</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💪 Step 8: Stress Test</h2>
</div>

In [ ]:
# Stress test: 3 tests x 3 models = 9 combinations
import re
import time
import concurrent.futures
from IPython.display import HTML, display

tests = [
    ("Simple", "What is 2+2? Answer with just the number.", 64),
    ("Medium", "List 5 common I2C bus errors and how to debug them.", 512),
    ("Creative", "Write a Python function that reads from a UART buffer and parses NMEA GPS sentences. Include docstring.", 1024),
]

def run_test(m, prompt, max_tokens):
    t0 = time.time()
    r = clients[m["label"]].chat.completions.create(
        model=m["model"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    elapsed = time.time() - t0
    tokens = r.usage.completion_tokens
    return {"label": m["label"], "color": m["color"], "tokens": tokens, "elapsed": elapsed, "tps": tokens / elapsed if elapsed > 0 else 0}

# Run all 9 combinations — each test group in parallel
all_results = {}
for test_name, prompt, max_tok in tests:
    print(f"Running: {test_name}...", flush=True)
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
        futures = {ex.submit(run_test, m, prompt, max_tok): m["label"] for m in MODELS}
        test_results = []
        for f in concurrent.futures.as_completed(futures):
            test_results.append(f.result())
    order = {"2B": 0, "35B": 1, "122B": 2}
    test_results.sort(key=lambda r: order.get(r["label"], 99))
    all_results[test_name] = test_results

# Display grouped tables
for test_name, prompt, max_tok in tests:
    results = all_results[test_name]
    rows = ""
    for r in results:
        rows += (
            f"<tr>"
            f"<td style='padding:6px 12px; color:{r['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['label']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['tokens']}</td>"
            f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['elapsed']:.1f}s</td>"
            f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['tps']:.1f}</td>"
            f"</tr>"
        )
    display(HTML(f"""
    <div style="color:#1e3a5f; font-weight:bold; margin:16px 0 4px 0;">{test_name} (max_tokens={max_tok})</div>
    <table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:4px 0 12px 0; font-family:monospace; min-width:400px;">
      <thead><tr style="background:#1e3a5f;">
        <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Tokens</th>
        <th style="padding:8px 12px; color:white; text-align:left;">Time</th>
        <th style="padding:8px 12px; color:white; text-align:left;">tok/s</th>
      </tr></thead>
      <tbody>{rows}</tbody>
    </table>
    """))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🗂️ Step 9: KV Cache — Why Second Turns Are Faster</h2>
</div>

<div class="alert alert-block alert-warning">
  <b>⚠️ Key Concept:</b> The KV (key-value) cache stores attention matrices from prior tokens so the model doesn’t recompute them on every turn. This speeds up multi-turn conversations but grows with context length — long conversations eat more RAM.
</div>

In [ ]:
# KV cache demo: cold vs cached timing for all 3 models
import time
from IPython.display import HTML, display

def kv_cache_test(m):
    """Test cold vs cached response time for a model."""
    messages = [{"role": "user", "content": "I'm going to ask you 3 questions about embedded systems. Ready?"}]

    # Turn 1 — cold context
    t0 = time.time()
    r = clients[m["label"]].chat.completions.create(
        model=m["model"], messages=messages, max_tokens=20,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    t1 = time.time()
    messages.append({"role": "assistant", "content": r.choices[0].message.content})

    # Turn 2 — reuses KV cache
    messages.append({"role": "user", "content": "Question 1: What's the difference between I2C and SPI?"})
    t2 = time.time()
    r2 = clients[m["label"]].chat.completions.create(
        model=m["model"], messages=messages, max_tokens=150,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    t3 = time.time()

    cold = t1 - t0
    cached = t3 - t2
    speedup = cold / cached if cached > 0 else 0
    return {"label": m["label"], "color": m["color"], "cold": cold, "cached": cached, "speedup": speedup}

results = []
for m in MODELS:
    results.append(kv_cache_test(m))

rows = ""
for r in results:
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{r['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['label']}</td>"
        f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['cold']:.2f}s</td>"
        f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{r['cached']:.2f}s</td>"
        f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['speedup']:.1f}x</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Turn 1 (cold)</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Turn 2 (cached)</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Speedup</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📝 Takeaways</h2>
</div>

- **Unified memory** lets Apple Silicon run models that don’t fit on discrete GPUs
- **MLX** serves models locally via an OpenAI-compatible API
- **4-bit quantization** shrinks models 8× with minimal quality loss
- **Temperature** controls randomness: 0 = deterministic, 1.5 = chaotic
- **System prompts** steer behavior without changing model weights
- **tok/s** is bounded by memory bandwidth, not compute
- **KV cache** speeds up multi-turn conversations by reusing prior computations
- **Running multiple models simultaneously** reveals the quality/speed tradeoff viscerally
- **Model size vs quality:** 0.8B is instant but shallow, 35B hits the sweet spot, 122B is best but slowest
- Everything here ran locally — no data left your machine

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 What’s Next</h2>
</div>

- **Episode 02:** Structured output — making LLMs return JSON you can actually parse
- **Episode 03:** Tool use — letting models call functions and interact with external systems
- **Episode 04:** RAG — grounding responses in your own documents
- **Episode 05:** Fine-tuning — teaching a model your domain